# 0. Install required python libraries

In [ ]:
!pip -qq install elasticsearch pandas datasets scikit-learn numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 979.4/979.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 1.7 MB/s eta 0:00:00


# 1. Connect to Elasticsearch
First, we need to establish a secure connection to your Elastic Serverless instance. We will use `getpass` to handle the API Key securely so it doesn't appear in your history.

In [ ]:
# !pip install elasticsearch pandas datasets numpy

from elasticsearch import Elasticsearch
from getpass import getpass
import numpy as np

print("--- Elastic Connection Setup ---")
ELASTIC_URL = input("Enter Elastic URL: ")
ELASTIC_API_KEY = getpass("Enter Elastic API Key: ")

client = Elasticsearch(
    hosts=ELASTIC_URL,
    api_key=ELASTIC_API_KEY,
    request_timeout=30
)

try:
    info = client.info()
    print(f"\n✅ Connected to Elastic")
except Exception as e:
    print(f"\n❌ Connection failed: {e}")

# 2. Load and Sample Data
We will use the **DBPedia-14** dataset (specifically the "Film" category) to create a realistic test corpus.


In [ ]:
import pandas as pd
from datasets import load_dataset

print("Loading DBPedia-14 dataset...")
ds = load_dataset("dbpedia_14", split="train")

# Filter for "Film" (Label 12)
df = pd.DataFrame(ds)
df = df[df['label'] == 12]

# We take an initial sample of 2,500 for the main tutorial
df_sample = df.sample(40000, random_state=42).reset_index(drop=True) #changed from 2500

print(f"✅ Dataset ready: {len(df_sample)} movie summaries.")

Loading DBPedia-14 dataset...
✅ Dataset ready: 40000 movie summaries.


# 3. Ingestion:

We will map the `content` field twice in the same index:
1.  **`content` (HNSW):** This uses the default `semantic_text` settings. It is optimized for speed and scale using the **HNSW** graph algorithm and likely uses **BBQ (Binary Quantization)**. This is the "Test Subject."
2.  **`content.raw` (Flat):** We explicitly configure this sub-field to use `type: "flat"`. This forces Elasticsearch to perform a **Brute-Force Exact Scan** of the full Float32 vectors. This is the "Control Group" (Ground Truth).

We will be using the **Jina Embedding V5 Text Model**

In [ ]:
from elasticsearch import helpers
import time

index_name = "movies-recall-demo"

# --- 1. Define Dual Mapping ---
mapping = {
    "properties": {
        "title": {"type": "text"},
        "content": {
            "type": "semantic_text",
            "inference_id": ".jina-embeddings-v5-text-small",
            "fields": {
                # The "Control Group": Forces exact brute-force scan
                "raw": {
                    "type": "semantic_text",
                    "inference_id": ".jina-embeddings-v5-text-small",
                    "index_options": {
                        "dense_vector": {
                            "type": "flat"
                        }
                    }
                }
            }
        }
    }
}

# --- 2. Reset Index ---
if client.indices.exists(index=index_name):
    client.indices.delete(index=index_name)
    print(f"♻️  Refreshing index '{index_name}'...")
else:
    print(f"🆕 Creating new index '{index_name}'...")

client.indices.create(index=index_name, mappings=mapping)
print(f"✅ Index ready with Flat + HNSW fields.")

# --- 3. Bulk Index ---
actions = []
print(f"Preparing {len(df_sample)} documents...")

for i, row in df_sample.iterrows():
    actions.append({
        "_index": index_name,
        "_id": str(i),
        "_source": {
            "title": row['title'],
            "content": row['content']
        }
    })

chunk_size = 500
print(f"🚀 Indexing documents (Chunk Size = {chunk_size})...")

success, errors = helpers.bulk(
    client, actions, chunk_size=chunk_size,
    raise_on_error=False, stats_only=False
)

if errors:
    print(f"\n⚠️ CRITICAL: {len(errors)} documents failed to index.")
    print(errors[0])
else:
    print(f"✅ Ingestion complete! Indexed {success} documents.")
    print("Waiting 10 seconds for inference to catch up...")
    time.sleep(10)

♻️  Refreshing index 'movies-recall-demo'...
✅ Index ready with Flat + HNSW fields.
Preparing 40000 documents...
🚀 Indexing documents (Chunk Size = 500)...
✅ Ingestion complete! Indexed 40000 documents.
Waiting 10 seconds for inference to catch up...
